1. 환경 설정 및 라이브러리 설치 (Cell 1)

In [1]:
import sys, transformers, pkgutil, importlib
print("transformers version:", transformers.__version__)
print("transformers path:", transformers.__file__)

# 충돌나는 모듈 확인
spec = pkgutil.find_loader("transformers.generation")
print("generation module:", spec.path if spec else None)


transformers version: 4.57.0
transformers path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
generation module: /usr/local/lib/python3.12/dist-packages/transformers/generation/__init__.py


/tmp/ipython-input-1373531468.py:6: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  spec = pkgutil.find_loader("transformers.generation")


In [2]:
!pip install transformers datasets sentencepiece accelerate torch

import datasets
from datasets import load_dataset, DatasetDict
from transformers import pipeline
import torch
import pandas as pd # 데이터 확인용

# GPU 사용 가능 여부 확인 및 설정 (Colab에서는 보통 GPU 사용 가능)
device = 0 if torch.cuda.is_available() else -1
print(f"사용 가능한 디바이스: {'GPU' if device == 0 else 'CPU'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.9.0
    Uninstalling fsspec-2025.9.0:
      Successfully uninstalled fsspec-2025.9.0
사용 가능한 디바이스: CPU


2. 데이터셋 로드 및 준비 (Cell 2)


In [3]:
ds = load_dataset("stanfordnlp/imdb")
print(ds)
print(ds["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [7]:
# 실습용 subset
num_samples_to_use = 200
imdb_subset = ds["train"].select(range(num_samples_to_use))

print("로드된 데이터셋 정보:")
print(imdb_subset)

print("\n첫 번째 데이터 예시 (ctext 확인):")
print(imdb_subset[0]['text'])

# 데이터셋 확인을 위해 Pandas DataFrame으로 변환 (선택 사항)
df_check = pd.DataFrame(imdb_subset)
print("\n데이터셋 일부 미리보기 (DataFrame):")
display(df_check.head(3)) # Colab 환경에서는 display()가 표 형태로 보여줍니다.

로드된 데이터셋 정보:
Dataset({
    features: ['text', 'label'],
    num_rows: 200
})

첫 번째 데이터 예시 (ctext 확인):
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0


3. 감정 분석 모델 파이프라인 로드

In [11]:
emotion_classifier = pipeline(
    task = "text-classification",
    model = "SamLowe/roberta-base-go_emotions",
    device = device,
    top_k = 1,
    truncation=True,     # 긴 입력 자르기
    padding=True,        # 배치 패딩
    max_length=512       # RoBERTa 최대 길이
)

print("감정 분석 파이프라인 로드 완료.")

# 감정 분석 테스트
test_emotion = emotion_classifier("This has to be the worst piece of garbage.")
print(f"감정 분석 테스트: {test_emotion}")

Device set to use cpu


감정 분석 파이프라인 로드 완료.
감정 분석 테스트: [[{'label': 'disgust', 'score': 0.8345435261726379}]]


4. 감정 분석 및 데이터셋에 추가

In [13]:
def analyze_emotion(example):
    out = emotion_classifier(example["text"])
    # 방어적 처리: top_k 사용 시 list[list[dict]], 미사용 시 list[dict]
    if isinstance(out, list) and len(out) > 0 and isinstance(out[0], list):
        label = out[0][0]["label"]
        score = out[0][0]["score"]
    else:
        label = out[0]["label"]
        score = out[0]["score"]
    return {"emotion": label, "emotion_score": float(score)}

print("감정 분석 작업을 시작합니다...")
final_dataset = imdb_subset.map(analyze_emotion)
print("감정 분석 작업 완료.")

print("\n최종 데이터셋 정보:")
print(final_dataset)

print("\n첫 번째 데이터의 영어 번역(english_summary)과 감정(emotion):")
print("--- 영어 번역 (English Summary) ---")
print(final_dataset[0]['text'])
print("\n--- 분석된 감정 (Emotion) ---")
print(final_dataset[0]['emotion'], f"(score={final_dataset[0]['emotion_score']:.4f})")

# 최종 데이터셋 확인 (Pandas)
df_final = pd.DataFrame(final_dataset)
print("\n최종 데이터셋 미리보기 (DataFrame):")
display(df_final) # 전체 선택된 샘플 표시

감정 분석 작업을 시작합니다...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

감정 분석 작업 완료.

최종 데이터셋 정보:
Dataset({
    features: ['text', 'label', 'emotion', 'emotion_score'],
    num_rows: 200
})

첫 번째 데이터의 영어 번역(english_summary)과 감정(emotion):
--- 영어 번역 (English Summary) ---
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, 

,text,label,emotion,emotion_score
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,neutral,0.383413
1,"""I Am Curious: Yellow"" is a risible and preten...",0,disapproval,0.546566
2,If only to avoid making this type of film in t...,0,neutral,0.622420
3,This film was probably inspired by Godard's Ma...,0,annoyance,0.529915
4,"Oh, brother...after hearing about this ridicul...",0,neutral,0.658857
...,...,...,...,...
195,This movie has beautiful scenery. Unfortunatel...,0,disapproval,0.439150
196,"Sorry, gave it a 1, which is the rating I give...",0,remorse,0.639858
197,I didn't know whether to laugh or cry at this ...,0,confusion,0.713716
198,The five or so really good westerns that Mann ...,0,admiration,0.366584


5. 결과 정리 및 마무리

In [14]:
df_text_emotion = df_final[['text', 'emotion']].copy()

print("새로운 DataFrame (text, emotion):")
display(df_text_emotion.head(5))

print("모든 작업이 완료되었습니다.")
# print("최종 데이터셋 컬럼:", final_dataset.column_names)

# 필요시 CSV 저장
# df_final.to_csv("klue_mrc_processed.csv", index=False, encoding='utf-8-sig')
# print("\n결과를 CSV 파일로 저장했습니다. (필요시 주석 해제)")

새로운 DataFrame (text, emotion):


,text,emotion
0,I rented I AM CURIOUS-YELLOW from my video sto...,neutral
1,"""I Am Curious: Yellow"" is a risible and preten...",disapproval
2,If only to avoid making this type of film in t...,neutral
3,This film was probably inspired by Godard's Ma...,annoyance
4,"Oh, brother...after hearing about this ridicul...",neutral


모든 작업이 완료되었습니다.
